In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/15 05:48:29 WARN Utils: Your hostname, codespaces-2bd11b, resolves to a loopback address: 127.0.0.1; using 10.0.1.73 instead (on interface eth0)
26/03/15 05:48:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/15 05:48:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Question 1. Install Spark and PySpark

In [3]:
spark.version

'4.1.1'

In [4]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-15 05:48:37--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.171.57.103, 3.171.57.69, 3.171.57.179, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.171.57.103|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet.1’

yellow_tripdata_202 100%[===================>]  67.84M   158MB/s    in 0.4s    

2026-03-15 05:48:38 (158 MB/s) - ‘yellow_tripdata_2025-11.parquet.1’ saved [71134255/71134255]



In [5]:
df_yellow = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [6]:
df_yellow = df_yellow.repartition(4)

Question 2. Yellow November 2025

In [7]:
df_yellow.write \
    .parquet('data/pq/yellow/2025/11', mode='overwrite')

In [8]:
df_yellow.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-07 15:04:17|  2025-11-07 15:39:15|              1|          7.3|         1|                 N|         262|    

Question 3. Count records

In [9]:
df_yellow \
    .filter('tpep_pickup_datetime >= "2025-11-15"') \
    .filter('tpep_pickup_datetime < "2025-11-16"') \
    .count()

162604

Question 4. Longest trip

In [10]:
from pyspark.sql import functions as F

# Use unix_timestamp to handle the conversion safely
df_yellow = df_yellow.withColumn('duration', 
    (F.unix_timestamp(F.col('tpep_dropoff_datetime')) - 
     F.unix_timestamp(F.col('tpep_pickup_datetime'))) / 3600
)

df_yellow.sort(F.col('duration').desc()).show(1)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|         duration|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------------+
|       2| 2025-11-26 20:22:12|  2025-11-30 15:01:00|              1|       

In [11]:
df_zones = spark.read.parquet('zones/')

In [12]:
df_yellow_w_zones = df_yellow \
    .join(df_zones, df_yellow.PULocationID == df_zones.LocationID)

Question 6. Least frequent pickup location zone

In [13]:
df_yellow_w_zones \
    .groupBy('zone') \
    .count() \
    .orderBy('count', ascending=True) \
    .limit(5) \
    .show()

+--------------------+-----+
|                zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|Eltingville/Annad...|    1|
|       Arden Heights|    1|
|       Port Richmond|    3|
| Green-Wood Cemetery|    4|
+--------------------+-----+

